# PF4 Quality-Control Summary

This notebook prepares a quality-control summary for the PF4 computational analysis.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
final_dir = Path("../data/final")
interim_dir = Path("../data/interim")

associations_path = final_dir / "pf4_cardiovascular_associations.csv"
variants_path = final_dir / "pf4_variants.csv"
hermes_ukb_path = interim_dir / "hermes" / "ukb_associations.csv"

out_qc_csv = final_dir / "pf4_qc_summary.csv"

In [3]:
associations_df = pd.read_csv(associations_path)

variants_df = pd.read_csv(variants_path) if variants_path.exists() else None
hermes_ukb_df = pd.read_csv(hermes_ukb_path) if hermes_ukb_path.exists() else None

print("Association table shape:", associations_df.shape)
print("Variant annotation table available:", variants_df is not None)
print("HERMES UKB-including table available:", hermes_ukb_df is not None)

Association table shape: (2614, 20)
Variant annotation table available: True
HERMES UKB-including table available: True


In [4]:
dataset_metadata = pd.DataFrame([
    {
        "source_dataset": "GWAS Catalog",
        "genome_build": "Mostly GRCh38 / source-dependent",
        "coordinates_original_or_lifted": "Original/source-provided",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Not applicable",
    },
    {
        "source_dataset": "HERMES",
        "genome_build": "GRCh37 / hg19",
        "coordinates_original_or_lifted": "Original HERMES coordinates",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Main table excludes UK Biobank",
    },
    {
        "source_dataset": "CARDIoGRAMplusC4D",
        "genome_build": "GRCh38",
        "coordinates_original_or_lifted": "Harmonized/source-provided coordinates",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "No UK Biobank-specific overlap flag used",
    },
    {
        "source_dataset": "FinnGen",
        "genome_build": "GRCh38",
        "coordinates_original_or_lifted": "Original FinnGen coordinates",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Independent replication resource",
    },
])

dataset_metadata

,source_dataset,genome_build,coordinates_original_or_lifted,both_GRCh37_and_GRCh38_available,sample_overlap_flag
0,GWAS Catalog,Mostly GRCh38 / source-dependent,Original/source-provided,No,Not applicable
1,HERMES,GRCh37 / hg19,Original HERMES coordinates,No,Main table excludes UK Biobank
2,CARDIoGRAMplusC4D,GRCh38,Harmonized/source-provided coordinates,No,No UK Biobank-specific overlap flag used
3,FinnGen,GRCh38,Original FinnGen coordinates,No,Independent replication resource


In [5]:
qc_df = associations_df.copy()

qc_df = qc_df.merge(
    dataset_metadata,
    on="source_dataset",
    how="left"
)

qc_df["p_value"] = pd.to_numeric(qc_df["p_value"], errors="coerce")
qc_df["effect_size"] = pd.to_numeric(qc_df["effect_size"], errors="coerce")
qc_df["standard_error"] = pd.to_numeric(qc_df["standard_error"], errors="coerce")
qc_df["odds_ratio"] = pd.to_numeric(qc_df["odds_ratio"], errors="coerce")
qc_df["allele_frequency"] = pd.to_numeric(qc_df["allele_frequency"], errors="coerce")

if "sample_size" in qc_df.columns:
    qc_df["sample_size"] = pd.to_numeric(qc_df["sample_size"], errors="coerce")

qc_df.shape

(2614, 24)

In [6]:
total_rows = len(qc_df)

def missing_result(column):
    if column not in qc_df.columns:
        return "column not available"
    missing = int(qc_df[column].isna().sum())
    present = int(qc_df[column].notna().sum())
    return f"{present}/{total_rows} rows available; {missing}/{total_rows} rows missing"

def pass_if_no_missing(column):
    if column not in qc_df.columns:
        return "Review"
    return "Pass" if qc_df[column].notna().all() else "Review"

def yes_no(value):
    return "Yes" if value else "No"

In [7]:
rows_by_source = (
    qc_df["source_dataset"]
    .value_counts(dropna=False)
    .reset_index()
)

rows_by_source.columns = ["source_dataset", "rows"]

rows_by_source

,source_dataset,rows
0,FinnGen,1941
1,CARDIoGRAMplusC4D,374
2,HERMES,192
3,GWAS Catalog,107


In [8]:
genome_build_summary = (
    qc_df
    .groupby(["source_dataset", "genome_build"], dropna=False)
    .size()
    .reset_index(name="rows")
)

genome_build_summary

,source_dataset,genome_build,rows
0,CARDIoGRAMplusC4D,GRCh38,374
1,FinnGen,GRCh38,1941
2,GWAS Catalog,Mostly GRCh38 / source-dependent,107
3,HERMES,GRCh37 / hg19,192


In [9]:
coordinate_handling_summary = (
    qc_df
    .groupby(["source_dataset", "coordinates_original_or_lifted"], dropna=False)
    .size()
    .reset_index(name="rows")
)

coordinate_handling_summary

,source_dataset,coordinates_original_or_lifted,rows
0,CARDIoGRAMplusC4D,Harmonized/source-provided coordinates,374
1,FinnGen,Original FinnGen coordinates,1941
2,GWAS Catalog,Original/source-provided,107
3,HERMES,Original HERMES coordinates,192


In [10]:
both_builds_summary = (
    qc_df
    .groupby(["source_dataset", "both_GRCh37_and_GRCh38_available"], dropna=False)
    .size()
    .reset_index(name="rows")
)

both_builds_summary

,source_dataset,both_GRCh37_and_GRCh38_available,rows
0,CARDIoGRAMplusC4D,No,374
1,FinnGen,No,1941
2,GWAS Catalog,No,107
3,HERMES,No,192


In [11]:
# Check repeated rsIDs within the same genome build for inconsistent positions.

position_check_df = qc_df.dropna(
    subset=["rsID", "genome_build", "chromosome", "position"]
).copy()

position_check_df["variant_position_key"] = (
    position_check_df["chromosome"].astype(str)
    + ":"
    + position_check_df["position"].astype(str)
)

position_concordance = (
    position_check_df
    .groupby(["rsID", "genome_build"])["variant_position_key"]
    .nunique()
    .reset_index(name="unique_positions_same_build")
)

position_conflicts = position_concordance[
    position_concordance["unique_positions_same_build"] > 1
]

position_conflicts.head()

,rsID,genome_build,unique_positions_same_build


In [12]:
# Check repeated rsIDs within the same genome build for inconsistent allele pairs.
# The allele pair is treated as unordered because effect allele direction can differ by dataset.

allele_check_df = qc_df.dropna(
    subset=["rsID", "genome_build", "effect_allele", "other_allele"]
).copy()

allele_check_df["allele_pair"] = allele_check_df.apply(
    lambda row: "/".join(sorted([
        str(row["effect_allele"]).upper(),
        str(row["other_allele"]).upper()
    ])),
    axis=1
)

allele_concordance = (
    allele_check_df
    .groupby(["rsID", "genome_build"])["allele_pair"]
    .nunique()
    .reset_index(name="unique_allele_pairs_same_build")
)

allele_conflicts = allele_concordance[
    allele_concordance["unique_allele_pairs_same_build"] > 1
]

allele_conflicts.head()

,rsID,genome_build,unique_allele_pairs_same_build
315,rs183028,GRCh38,2
342,rs187727263,GRCh38,2
556,rs535524592,GRCh38,2


In [13]:
zero_pvalue_rows = qc_df[qc_df["p_value"] == 0].copy()

zero_pvalue_rows.shape

(2, 24)

In [14]:
fully_duplicated_rows = int(qc_df[associations_df.columns].duplicated().sum())

repeated_rsid_summary = (
    qc_df
    .dropna(subset=["rsID"])
    .groupby("rsID")
    .agg(
        total_rows=("rsID", "size"),
        source_datasets=("source_dataset", lambda x: "; ".join(sorted(set(x.dropna())))),
        phenotypes=("phenotype", lambda x: "; ".join(sorted(set(x.dropna())))),
    )
    .reset_index()
)

repeated_rsid_summary = repeated_rsid_summary[
    repeated_rsid_summary["total_rows"] > 1
].sort_values("total_rows", ascending=False)

fully_duplicated_rows, len(repeated_rsid_summary)

(0, 612)

In [15]:
if variants_df is not None and {"MAX_freq", "Priority"}.issubset(variants_df.columns):
    missing_max_freq_df = variants_df[variants_df["MAX_freq"].isna()].copy()

    missing_max_freq_rows = int(len(missing_max_freq_df))
    missing_max_freq_below_threshold_rows = int(
        (missing_max_freq_df["Priority"] == "Below threshold").sum()
    )
    missing_max_freq_unknown_rows = int(
        (missing_max_freq_df["Priority"] == "Unknown").sum()
    )
else:
    missing_max_freq_rows = None
    missing_max_freq_below_threshold_rows = None
    missing_max_freq_unknown_rows = None

missing_max_freq_rows, missing_max_freq_below_threshold_rows, missing_max_freq_unknown_rows

(520, 520, 0)

In [16]:
gwas_df = qc_df[qc_df["source_dataset"] == "GWAS Catalog"].copy()

if "is_selected_trait" in gwas_df.columns:
    direct_cardiovascular_trait_rows = int(gwas_df["is_selected_trait"].fillna(False).sum())
    broader_trait_rows = int((~gwas_df["is_selected_trait"].fillna(False)).sum())
else:
    direct_cardiovascular_trait_rows = 0
    broader_trait_rows = int(len(gwas_df))

direct_cardiovascular_trait_rows, broader_trait_rows

/var/folders/ml/rhzbx8mx0jj25ftqqhfslrkc0000gn/T/ipykernel_88113/4289368416.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  direct_cardiovascular_trait_rows = int(gwas_df["is_selected_trait"].fillna(False).sum())
/var/folders/ml/rhzbx8mx0jj25ftqqhfslrkc0000gn/T/ipykernel_88113/4289368416.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  broader_trait_rows = int((~gwas_df["is_selected_trait"].fillna(False)).sum())


(0, 107)

In [17]:
hermes_main_rows = int((qc_df["source_dataset"] == "HERMES").sum())
hermes_ukb_file_available = hermes_ukb_df is not None
hermes_ukb_rows = int(len(hermes_ukb_df)) if hermes_ukb_df is not None else 0

hermes_main_rows, hermes_ukb_file_available, hermes_ukb_rows

(192, True, 191)

In [18]:
qc_df["calculated_odds_ratio"] = np.exp(qc_df["effect_size"])

qc_df["or_absolute_difference"] = (
    qc_df["odds_ratio"] - qc_df["calculated_odds_ratio"]
).abs()

rows_with_effect_size = int(qc_df["effect_size"].notna().sum())
rows_with_odds_ratio = int(qc_df["odds_ratio"].notna().sum())
rows_with_effect_size_but_missing_odds_ratio = int(
    ((qc_df["effect_size"].notna()) & (qc_df["odds_ratio"].isna())).sum()
)

max_or_difference = qc_df["or_absolute_difference"].max(skipna=True)

rows_with_effect_size, rows_with_odds_ratio, rows_with_effect_size_but_missing_odds_ratio, max_or_difference

(2507, 2507, 0, np.float64(4.98901378431782e-07))

In [19]:
qc_rows = [
    {
        "qc_check": "Source dataset recorded for each row",
        "status": "Pass" if qc_df["source_dataset"].notna().all() else "Review",
        "result": f"All {total_rows} association rows have source_dataset recorded"
        if qc_df["source_dataset"].notna().all()
        else f"{int(qc_df['source_dataset'].isna().sum())}/{total_rows} rows missing source_dataset",
    },
    {
        "qc_check": "Genome build used by each dataset recorded",
        "status": "Pass" if qc_df["genome_build"].notna().all() else "Review",
        "result": f"All {total_rows} association rows have genome_build recorded"
        if qc_df["genome_build"].notna().all()
        else f"{int(qc_df['genome_build'].isna().sum())}/{total_rows} rows missing genome_build",
    },
    {
        "qc_check": "Coordinates recorded as original or lifted over",
        "status": "Pass" if qc_df["coordinates_original_or_lifted"].notna().all() else "Review",
        "result": f"All {total_rows} association rows have coordinate handling recorded"
        if qc_df["coordinates_original_or_lifted"].notna().all()
        else f"{int(qc_df['coordinates_original_or_lifted'].isna().sum())}/{total_rows} rows missing coordinate handling",
    },
    {
        "qc_check": "Availability of both GRCh37 and GRCh38 coordinates recorded",
        "status": "Pass" if qc_df["both_GRCh37_and_GRCh38_available"].notna().all() else "Review",
        "result": "; ".join(
            both_builds_summary.apply(
                lambda row: f"{row['source_dataset']}: {row['both_GRCh37_and_GRCh38_available']}",
                axis=1
            )
        ),
    },
    {
        "qc_check": "rsIDs, alleles, and positions checked for availability and concordance",
        "status": "Pass" if len(position_conflicts) == 0 and len(allele_conflicts) == 0 else "Review",
        "result": (
            f"{int(qc_df['rsID'].isna().sum())} rows missing rsID; "
            f"{int(qc_df['chromosome'].isna().sum())} rows missing chromosome; "
            f"{int(qc_df['position'].isna().sum())} rows missing position; "
            f"{int(qc_df['effect_allele'].isna().sum())} rows missing effect_allele; "
            f"{int(qc_df['other_allele'].isna().sum())} rows missing other_allele; "
            f"{len(position_conflicts)} rsID-position conflicts within same build; "
            f"{len(allele_conflicts)} rsID-allele-pair conflicts within same build"
        ),
    },
    {
        "qc_check": "Effect allele and non-effect allele clearly defined",
        "status": (
            "Pass"
            if qc_df["effect_allele"].notna().all() and qc_df["other_allele"].notna().all()
            else "Review"
        ),
        "result": (
            f"effect_allele: {int(qc_df['effect_allele'].notna().sum())}/{total_rows} available, "
            f"{int(qc_df['effect_allele'].isna().sum())} missing; "
            f"other_allele: {int(qc_df['other_allele'].notna().sum())}/{total_rows} available, "
            f"{int(qc_df['other_allele'].isna().sum())} missing"
        ),
    },
    {
        "qc_check": "p-values, effect sizes, standard errors, allele frequencies, and sample sizes available",
        "status": "Pass with note" if "sample_size" not in qc_df.columns else "Review",
        "result": (
            f"p_value: {missing_result('p_value')}; "
            f"effect_size: {missing_result('effect_size')}; "
            f"standard_error: {missing_result('standard_error')}; "
            f"allele_frequency: {missing_result('allele_frequency')}; "
            f"sample_size: {missing_result('sample_size')}"
        ),
    },
    {
        "qc_check": "p-values reported as zero checked",
        "status": "Pass" if len(zero_pvalue_rows) == 0 else "Review",
        "result": f"{len(zero_pvalue_rows)} rows with p_value = 0",
    },
    {
        "qc_check": "Same variant appearing multiple times across traits or datasets checked",
        "status": "Expected" if len(repeated_rsid_summary) > 0 else "Pass",
        "result": f"{len(repeated_rsid_summary)} rsIDs appear in more than one association row",
    },
    {
        "qc_check": "Fully duplicated rows checked",
        "status": "Pass" if fully_duplicated_rows == 0 else "Review",
        "result": f"{fully_duplicated_rows} fully duplicated rows",
    },
    {
        "qc_check": "Missing allele frequencies classified as Unknown rather than automatically Below threshold",
        "status": (
            "Pass"
            if missing_max_freq_rows is not None and missing_max_freq_below_threshold_rows == 0
            else "Review"
        ),
        "result": (
            f"{missing_max_freq_rows} variants missing MAX_freq; "
            f"{missing_max_freq_unknown_rows} classified as Unknown; "
            f"{missing_max_freq_below_threshold_rows} classified as Below threshold"
            if missing_max_freq_rows is not None
            else "variant annotation table or required columns not available"
        ),
    },
    {
        "qc_check": "GWAS Catalog associations checked for direct cardiovascular traits or broader molecular/biomarker traits",
        "status": "Pass",
        "result": (
            f"{direct_cardiovascular_trait_rows} direct cardiovascular GWAS Catalog rows; "
            f"{broader_trait_rows} broader or non-selected GWAS Catalog rows"
        ),
    },
    {
        "qc_check": "HERMES results checked as excluding or including UK Biobank",
        "status": "Pass" if hermes_main_rows > 0 else "Review",
        "result": (
            f"Main table contains {hermes_main_rows} HERMES rows from the UKB-excluding version; "
            f"separate UKB-including HERMES file "
            f"{'found' if hermes_ukb_file_available else 'not found'} "
            f"with {hermes_ukb_rows} rows"
        ),
    },
    {
        "qc_check": "Potential sample overlap clearly flagged",
        "status": "Pass" if qc_df["sample_overlap_flag"].notna().all() else "Review",
        "result": (
            "HERMES: main table excludes UK Biobank; "
            "FinnGen: independent replication resource; "
            "CARDIoGRAMplusC4D: no UKB-specific flag; "
            "GWAS Catalog: not applicable"
        ),
    },
    {
        "qc_check": "Odds ratios available or derivable from effect size",
        "status": "Pass" if rows_with_effect_size_but_missing_odds_ratio == 0 else "Review",
        "result": (
            f"{rows_with_effect_size} rows with effect_size; "
            f"{rows_with_odds_ratio} rows with odds_ratio; "
            f"{rows_with_effect_size_but_missing_odds_ratio} rows with effect_size but missing odds_ratio; "
            f"maximum OR difference from exp(effect_size): {max_or_difference}"
        ),
    },
]

qc_summary_df = pd.DataFrame(qc_rows)

qc_summary_df

,qc_check,status,result
0,Source dataset recorded for each row,Pass,All 2614 association rows have source_dataset ...
1,Genome build used by each dataset recorded,Pass,All 2614 association rows have genome_build re...
2,Coordinates recorded as original or lifted over,Pass,All 2614 association rows have coordinate hand...
3,Availability of both GRCh37 and GRCh38 coordin...,Pass,CARDIoGRAMplusC4D: No; FinnGen: No; GWAS Catal...
4,"rsIDs, alleles, and positions checked for avai...",Review,114 rows missing rsID; 107 rows missing chromo...
5,Effect allele and non-effect allele clearly de...,Review,"effect_allele: 2614/2614 available, 0 missing;..."
6,"p-values, effect sizes, standard errors, allel...",Pass with note,p_value: 2614/2614 rows available; 0/2614 rows...
7,p-values reported as zero checked,Review,2 rows with p_value = 0
8,Same variant appearing multiple times across t...,Expected,612 rsIDs appear in more than one association row
9,Fully duplicated rows checked,Pass,0 fully duplicated rows


In [20]:
qc_summary_df.to_csv(out_qc_csv, index=False)

print("Saved QC summary to:", out_qc_csv)

Saved QC summary to: ../data/final/pf4_qc_summary.csv
